# Compare embeddings across extractions

Compares per-cell embeddings across one or more `extract_embeddings.py` runs --
originally written to line up `PhikonV2_448_h5` (unmasked) against
`PhikonV2_448_masked_h5` (central 3x3 tokens masked), but every section below is
driven off the `EXTRACTIONS` / `COMPARISONS` / `REDUCTION_TARGETS` config lists
defined a couple of cells down, so adding another model, another token, or
another pairwise comparison is a one-line edit -- no other code needs to change.
See the **Extending this notebook** section at the bottom for the exact recipe.

For each extraction this loads, per cell:
- `cls` -- the CLS token (`embeddings_cls`).
- `central` -- **not** an h5 dataset on its own: the mean of the 4 central patch
  tokens PhikonV2InferenceProvider saves by default (`embeddings_top_left` /
  `top_right` / `bottom_left` / `bottom_right` -- see
  `src/python/extract_embeddings/inference_providers/phikon_v2_inference_provider.py`),
  averaged together by `add_central_token()` below.

Then:
1. **Cosine similarity / drift** between configurable pairs of (extraction, token)
   -- e.g. CLS vs. central, masked CLS vs. CLS, masked central vs. CLS -- with a
   per-cell-type breakdown and a look at the highest-drift cells.
2. **PCA + UMAP**, colored by H&E slide of origin (`wsi`) and by mapped cell type
   (`cell_type`), for each (extraction, token) individually, plus one joint
   projection across all of them together (colored by source) to see how the
   different extractions sit relative to each other in embedding space.

In [ ]:
import os
import sys
from pathlib import Path

REPO_ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
sys.path.insert(0, str(REPO_ROOT))


def _load_dotenv(path: Path) -> None:
    """Minimal `.env` loader (KEY=value per line) -- avoids an extra dependency.
    Mirrors how every other entry point in this repo expects env vars to be set
    (see src/python/code_configs/paths.py)."""
    if not path.exists():
        return
    for line in path.read_text().splitlines():
        line = line.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        key, _, value = line.partition("=")
        key, value = key.strip(), value.strip()
        if value:
            os.environ.setdefault(key, value)


_load_dotenv(REPO_ROOT / ".env")

In [ ]:
import h5py
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA

from src.python.code_configs import paths
from src.python.code_configs.mappings import mapping_factory

try:
    import umap
    _HAVE_UMAP = True
except ImportError:
    _HAVE_UMAP = False
    print("umap-learn is not installed (`pip install umap-learn`) -- the UMAP panels "
          "below will be skipped; PCA panels still work without it.")

pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 30)

## Config -- extractions to compare

`DATASETS_ROOT` / `{model_name}_h5/{wsi}/embeddings_dataset.h5` is the layout
`extract_embeddings.py` writes and `configs/*.yaml`'s `data.base_dir` reads from
(see `.env.example`). Each entry in `EXTRACTIONS` is one such `{model_name}_h5`
root plus which token datasets to load from it.

**To add another model or another extraction of the same model**, add a new key
here -- the loading cell below picks it up automatically, and you can then
reference `("your_new_key", "cls")` / `("your_new_key", "central")` etc. anywhere
a `(extraction, token)` pair is expected further down (`COMPARISONS`,
`REDUCTION_TARGETS`, `JOINT_TARGETS`).

Directory names below match what was given for this run -- if your actual h5
roots instead carry the literal `extract_embeddings.py` output suffix (e.g.
`PhikonV2_448_224_h5` / `PhikonV2_448_224_masked_h5`, see that file's
`build_resized_cell_configs(..., output_suffix=...)` calls), just edit the two
`h5_root` lines below.

In [ ]:
DATASETS_ROOT = Path(paths.DATASETS_ROOT)

MAPPING_NAME = "simplified_broad"  # any key in src/python/code_configs/mappings.py
MAPPING = mapping_factory(MAPPING_NAME)

SEED = 42

# Every key in `corner_keys` is the suffix after "embeddings_" in
# embeddings_dataset.h5 (see InferenceProvider.create_output_file / save_embeddings).
# "central" is not an h5 dataset -- it's added by add_central_token() below as the
# mean of `corner_keys`.
EXTRACTIONS = {
    "phikon_v2": dict(
        h5_root=DATASETS_ROOT / "PhikonV2_448_h5",
        corner_keys=["top_left", "top_right", "bottom_left", "bottom_right"],
        load_cls=True,
        wsi_list=None,  # None = auto-discover every subdir with an embeddings_dataset.h5
    ),
    "phikon_v2_masked": dict(
        h5_root=DATASETS_ROOT / "PhikonV2_448_masked_h5",
        corner_keys=["top_left", "top_right", "bottom_left", "bottom_right"],
        load_cls=True,
        wsi_list=None,
    ),
    # Add another model/extraction here, e.g.:
    # "uni2_448": dict(h5_root=DATASETS_ROOT / "UNI2_448_h5",
    #                   corner_keys=["top_left", "top_right", "bottom_left", "bottom_right"],
    #                   load_cls=True, wsi_list=None),
}

## Loading

`load_tokens` reads one `embeddings_dataset.h5` per WSI subdirectory and
concatenates across WSIs; `cell_labels` are looked up through `MAPPING` exactly
like `src/python/utils/loading_functions.py` does for the probing/classifier
pipelines (mapping dict keys are raw `bytes`, matching what h5py hands back for
a variable-length utf-8 dataset read without `.asstr()`).

In [ ]:
def discover_wsis(h5_root: Path) -> list[str]:
    """Every subdirectory of `h5_root` holding an embeddings_dataset.h5."""
    return sorted(p.name for p in Path(h5_root).iterdir() if (p / "embeddings_dataset.h5").is_file())


def load_tokens(h5_root: Path, token_keys: list[str], wsi_list: list[str] | None = None,
                 mapping: dict | None = None) -> tuple[dict[str, np.ndarray], pd.DataFrame]:
    """Load `embeddings_{key}` for every key in `token_keys`, for every WSI in
    `wsi_list` (or every WSI auto-discovered under `h5_root` if None) under `h5_root`.

    Returns:
        tokens: {token_key: (N, D) float32 array}, same row order as `meta`.
        meta: DataFrame with columns cell_id, wsi, cell_label_raw, cell_type --
            cell_type is `cell_label_raw` passed through `mapping` (or identical to
            cell_label_raw if mapping is None).
    """
    h5_root = Path(h5_root)
    if wsi_list is None:
        wsi_list = discover_wsis(h5_root)

    tokens: dict[str, list[np.ndarray]] = {k: [] for k in token_keys}
    meta_rows = []
    for wsi in wsi_list:
        h5_path = h5_root / wsi / "embeddings_dataset.h5"
        with h5py.File(h5_path, "r") as f:
            cell_ids = f["cell_ids"][()]
            cell_labels = f["cell_labels"][()]  # raw bytes -- matches mappings.py's bytes-keyed dicts
            for key in token_keys:
                dset_name = f"embeddings_{key}"
                if dset_name not in f:
                    available = sorted(k for k in f if k.startswith("embeddings_"))
                    raise KeyError(f"{dset_name!r} not found in {h5_path} -- available: {available}")
                tokens[key].append(f[dset_name][()])

        cell_ids_dec = [c.decode("utf-8") if isinstance(c, bytes) else c for c in cell_ids]
        cell_label_raw = [l.decode("utf-8") if isinstance(l, bytes) else l for l in cell_labels]
        if mapping is not None:
            cell_type = [mapping.get(l, "Unknown") for l in cell_labels]
        else:
            cell_type = list(cell_label_raw)
        meta_rows.append(pd.DataFrame({
            "cell_id": cell_ids_dec, "wsi": wsi,
            "cell_label_raw": cell_label_raw, "cell_type": cell_type,
        }))

    meta = pd.concat(meta_rows, ignore_index=True)
    tokens = {k: np.concatenate(v, axis=0) for k, v in tokens.items()}
    return tokens, meta


def add_central_token(tokens: dict[str, np.ndarray], corner_keys: list[str],
                       out_key: str = "central") -> dict[str, np.ndarray]:
    """Mean-pool `corner_keys` into a new `out_key` entry (doesn't mutate `tokens`)."""
    tokens = dict(tokens)
    tokens[out_key] = np.stack([tokens[k] for k in corner_keys], axis=0).mean(axis=0)
    return tokens

In [ ]:
EXTRACTIONS_DATA = {}
for name, cfg in EXTRACTIONS.items():
    token_keys = list(cfg["corner_keys"]) + (["cls"] if cfg["load_cls"] else [])
    tokens, meta = load_tokens(cfg["h5_root"], token_keys, wsi_list=cfg["wsi_list"], mapping=MAPPING)
    tokens = add_central_token(tokens, cfg["corner_keys"])
    EXTRACTIONS_DATA[name] = dict(tokens=tokens, meta=meta)
    print(f"[{name}] {len(meta):,} cells across {meta['wsi'].nunique()} WSI(s) -- "
          f"tokens={sorted(tokens)}; cell types={sorted(meta['cell_type'].unique())}")

## Sanity check: masked central tokens

PhikonV2's masked run both greys out the central pixels *and* (via
`InferenceProvider.register_mask_token_hook`) overwrites the central 3x3 patch
tokens' embeddings with the model's own learned `mask_token`, **in embedding
space, unconditionally** -- i.e. regardless of what the (greyed-out) pixels
actually were. `top_left`/`top_right`/`bottom_left`/`bottom_right` all fall
inside that masked 3x3 block for PhikonV2 (grid positions {5,6,7}x{5,6,7} on a
14x14 grid; `patches_to_save` defaults to (6,6)/(6,7)/(7,6)/(7,7)), so
`central` in the `*_masked` extraction should come out **identical for every
cell** -- carrying no per-cell signal at all. Worth confirming before reading
too much into the "masked central vs CLS" comparison below: any per-cell
variation there is coming entirely from the CLS side, not from `central`.

In [ ]:
_central_masked = EXTRACTIONS_DATA["phikon_v2_masked"]["tokens"]["central"]
_per_dim_std = _central_masked.std(axis=0).mean()
_scale = np.abs(_central_masked).mean()
print(f"mean per-dimension std of 'central' across cells in phikon_v2_masked: {_per_dim_std:.6f} "
      f"(mean |value| = {_scale:.4f})")
if _per_dim_std < 1e-3 * max(_scale, 1e-8):
    print("-> effectively constant across cells, confirming the note above: 'central' in the masked "
          "extraction is just PhikonV2's mask_token, not a content-dependent embedding.")
else:
    print("-> varies across cells more than expected from a pure mask_token overwrite -- re-check "
          "the masking config (mask_grid_size / mask_token_size / patches_to_save) for this run.")

## Cosine similarity / drift between token pairs

`get_aligned` row-aligns any two `(extraction, token)` pairs on shared
`(wsi, cell_id)` -- `phikon_v2` / `phikon_v2_masked` happen to iterate the exact
same cells in the exact same order (both built from the same
`patch_coordinates.h5` via `ResizedCellDataset`), but aligning explicitly keeps
this correct even if you add a model/extraction with different cell coverage.

**To add another comparison**, append a dict to `COMPARISONS` with any two
`(extraction, token)` pairs already present in `EXTRACTIONS_DATA`.

In [ ]:
def row_cosine_similarity(a: np.ndarray, b: np.ndarray) -> np.ndarray:
    a = a.astype(np.float64)
    b = b.astype(np.float64)
    num = np.einsum("ij,ij->i", a, b)
    denom = np.linalg.norm(a, axis=1) * np.linalg.norm(b, axis=1)
    return num / np.clip(denom, 1e-12, None)


def get_aligned(extraction_a: str, token_a: str, extraction_b: str, token_b: str):
    """Row-align (extraction_a, token_a) against (extraction_b, token_b) on shared
    (wsi, cell_id). Returns (A, B, meta) -- A/B are (N, D) arrays in the same cell
    order as `meta` (metadata columns taken from extraction_a)."""
    meta_a = EXTRACTIONS_DATA[extraction_a]["meta"]
    meta_b = EXTRACTIONS_DATA[extraction_b]["meta"]
    idx_a = meta_a.reset_index().rename(columns={"index": "_ia"})
    idx_b = meta_b.reset_index().rename(columns={"index": "_ib"})[["wsi", "cell_id", "_ib"]]
    merged = idx_a.merge(idx_b, on=["wsi", "cell_id"], how="inner")
    if len(merged) < min(len(meta_a), len(meta_b)):
        print(f"WARNING: only {len(merged):,}/{min(len(meta_a), len(meta_b)):,} cells matched between "
              f"({extraction_a}) and ({extraction_b}) on (wsi, cell_id) -- check wsi_list / cell coverage.")
    a = EXTRACTIONS_DATA[extraction_a]["tokens"][token_a][merged["_ia"].to_numpy()]
    b = EXTRACTIONS_DATA[extraction_b]["tokens"][token_b][merged["_ib"].to_numpy()]
    return a, b, merged.drop(columns=["_ia", "_ib"]).reset_index(drop=True)


def run_comparison(name: str, a: tuple[str, str], b: tuple[str, str], top_n: int = 15, plot: bool = True):
    """Cosine similarity / drift (1 - cosine) between a=(extraction, token) and
    b=(extraction, token), with a per-cell-type breakdown and the top_n
    highest-drift cells. Returns (df, by_type)."""
    a_ext, a_tok = a
    b_ext, b_tok = b
    A, B, meta = get_aligned(a_ext, a_tok, b_ext, b_tok)
    cos = row_cosine_similarity(A, B)

    df = meta.copy()
    df["cosine_similarity"] = cos
    df["drift"] = 1 - cos

    print(f"\n=== {name} ===  ({a_ext}/{a_tok}  vs  {b_ext}/{b_tok})  n={len(df):,}")
    print(f"cosine similarity: mean={cos.mean():.4f}  std={cos.std():.4f}  "
          f"min={cos.min():.4f}  max={cos.max():.4f}")

    by_type = (df.groupby("cell_type")["drift"]
                 .agg(["mean", "median", "std", "count"])
                 .sort_values("mean", ascending=False))
    print(by_type.round(4))

    top = df.sort_values("drift", ascending=False).head(top_n)
    print(f"\ntop {top_n} highest-drift cells:")
    print(top[["cell_id", "wsi", "cell_type", "cosine_similarity", "drift"]].to_string(index=False))

    if plot:
        fig, axes = plt.subplots(1, 2, figsize=(11, 4))
        axes[0].hist(cos, bins=60, color="#4C72B0")
        axes[0].set_title(f"{name}\ncosine similarity")
        axes[0].set_xlabel("cosine similarity")
        axes[0].set_ylabel("cells")

        order = by_type.index
        axes[1].bar(order, by_type["mean"], yerr=by_type["std"], color="#DD8452")
        axes[1].set_title("mean drift (1 - cosine) by cell type")
        axes[1].set_ylabel("drift")
        axes[1].tick_params(axis="x", rotation=45)
        for label in axes[1].get_xticklabels():
            label.set_ha("right")
        fig.tight_layout()
        plt.show()

    return df, by_type

In [ ]:
COMPARISONS = [
    dict(name="CLS vs central token (unmasked)",
         a=("phikon_v2", "cls"), b=("phikon_v2", "central")),
    dict(name="masked CLS vs CLS (unmasked)",
         a=("phikon_v2_masked", "cls"), b=("phikon_v2", "cls")),
    dict(name="masked central vs CLS (unmasked)",
         a=("phikon_v2_masked", "central"), b=("phikon_v2", "cls")),
    dict(name="masked CLS vs central token (unmasked)",
         a=("phikon_v2_masked", "cls"), b=("phikon_v2", "central")),
    # Add another pairing here -- `a`/`b` are any (extraction, token) pair
    # registered in EXTRACTIONS_DATA above.
]

COMPARISON_RESULTS = {}
for cmp in COMPARISONS:
    df, by_type = run_comparison(cmp["name"], cmp["a"], cmp["b"])
    COMPARISON_RESULTS[cmp["name"]] = dict(df=df, by_type=by_type)

## Interpretation: which cells drift the most

Focused side-by-side on the two comparisons explicitly called out up front --
**CLS vs central** (both unmasked: how much does a cell's own local central
patch already agree with its global CLS summary, baseline) and **masked CLS vs
central** (how much does knocking out the central tokens move the global CLS
away from where the (unmasked) central patch used to sit). Per-cell-type mean
drift for every comparison is already printed above (`COMPARISON_RESULTS[name]["by_type"]`)
-- this just pulls those two out together and lists the individual
highest-drift cells so you can go look at them (by `cell_id` / `wsi`) if you
want to inspect the actual patches.

In [ ]:
HIGHLIGHT_COMPARISONS = [
    "CLS vs central token (unmasked)",
    "masked CLS vs central token (unmasked)",
]

fig, axes = plt.subplots(1, len(HIGHLIGHT_COMPARISONS), figsize=(6 * len(HIGHLIGHT_COMPARISONS), 4), sharey=True)
axes = np.atleast_1d(axes)
for ax, name in zip(axes, HIGHLIGHT_COMPARISONS):
    by_type = COMPARISON_RESULTS[name]["by_type"]
    order = by_type.index
    ax.bar(order, by_type["mean"], yerr=by_type["std"], color="#55A868")
    ax.set_title(name, fontsize=10)
    ax.set_ylabel("mean drift (1 - cosine)")
    ax.tick_params(axis="x", rotation=45)
    for label in ax.get_xticklabels():
        label.set_ha("right")
fig.tight_layout()
plt.show()

for name in HIGHLIGHT_COMPARISONS:
    by_type = COMPARISON_RESULTS[name]["by_type"]
    top_type, low_type = by_type["mean"].idxmax(), by_type["mean"].idxmin()
    print(f"[{name}] highest mean drift: {top_type} ({by_type.loc[top_type, 'mean']:.3f}); "
          f"lowest: {low_type} ({by_type.loc[low_type, 'mean']:.3f})")
    top_cells = COMPARISON_RESULTS[name]["df"].sort_values("drift", ascending=False).head(10)
    print(top_cells[["cell_id", "wsi", "cell_type", "drift"]].to_string(index=False))
    print()

## PCA + UMAP, per (extraction, token)

For each entry in `REDUCTION_TARGETS`: PCA (direct top-2 PCs) and UMAP (fit on
the embeddings, pre-reduced via PCA to `pca_components_for_umap` dims first for
speed -- standard practice on high-dim ViT embeddings), each colored by `wsi`
(H&E slide of origin) and by `cell_type` (mapped cell type). Subsampled to
`N_SUBSAMPLE_PLOTS` cells for speed/readability -- set to `None` to use every
cell.

**To add another panel**, append an `(extraction, token)` pair to
`REDUCTION_TARGETS`.

In [ ]:
N_SUBSAMPLE_PLOTS = 20_000


def subsample_index(n_total: int, n: int | None, seed: int = SEED) -> np.ndarray:
    if n is None or n >= n_total:
        return np.arange(n_total)
    rng = np.random.default_rng(seed)
    return rng.choice(n_total, size=n, replace=False)


def reduce_2d(embeddings: np.ndarray, seed: int = SEED, pca_components_for_umap: int = 50):
    """Returns (pca_coords, umap_coords, pca_explained_variance_ratio).
    umap_coords is None if umap-learn isn't installed."""
    pca = PCA(n_components=2, random_state=seed).fit(embeddings)
    pca_coords = pca.transform(embeddings)

    umap_coords = None
    if _HAVE_UMAP:
        n_pre = min(pca_components_for_umap, embeddings.shape[0] - 1, embeddings.shape[1])
        pre = (PCA(n_components=n_pre, random_state=seed).fit_transform(embeddings)
               if n_pre < embeddings.shape[1] else embeddings)
        umap_coords = umap.UMAP(n_components=2, random_state=seed).fit_transform(pre)

    return pca_coords, umap_coords, pca.explained_variance_ratio_


def scatter_colored(ax, coords: np.ndarray, labels: pd.Series, title: str,
                     cmap_name: str = "tab20", s: float = 5, alpha: float = 0.6) -> None:
    cats = pd.Categorical(labels)
    cmap = plt.get_cmap(cmap_name, max(len(cats.categories), 1))
    ax.scatter(coords[:, 0], coords[:, 1], c=cats.codes, cmap=cmap, s=s, alpha=alpha, linewidths=0)
    ax.set_title(title, fontsize=9)
    ax.set_xticks([])
    ax.set_yticks([])
    handles = [plt.Line2D([0], [0], marker="o", linestyle="", color=cmap(i), label=str(cat))
               for i, cat in enumerate(cats.categories)]
    ax.legend(handles=handles, bbox_to_anchor=(1.02, 1), loc="upper left", fontsize=6, frameon=False)


def plot_reduction_grid(extraction: str, token: str, n_subsample: int | None = N_SUBSAMPLE_PLOTS,
                         color_by: tuple[str, ...] = ("wsi", "cell_type")) -> None:
    data = EXTRACTIONS_DATA[extraction]
    meta_full = data["meta"]
    idx = subsample_index(len(meta_full), n_subsample)
    emb = data["tokens"][token][idx]
    meta = meta_full.iloc[idx].reset_index(drop=True)

    pca_coords, umap_coords, var_ratio = reduce_2d(emb)
    rows = [("PCA", pca_coords)] + ([("UMAP", umap_coords)] if umap_coords is not None else [])

    fig, axes = plt.subplots(len(rows), len(color_by), figsize=(5 * len(color_by), 4.5 * len(rows)), squeeze=False)
    for r, (method, coords) in enumerate(rows):
        for c, color_key in enumerate(color_by):
            extra = f" (PC1+2, {var_ratio[:2].sum():.1%} var)" if method == "PCA" else ""
            scatter_colored(axes[r][c], coords, meta[color_key], f"{method} by {color_key}{extra}")
    fig.suptitle(f"{extraction} / {token}  (n={len(meta):,} cells)")
    fig.tight_layout()
    plt.show()

In [ ]:
REDUCTION_TARGETS = [
    ("phikon_v2", "cls"),
    ("phikon_v2", "central"),
    ("phikon_v2_masked", "cls"),
    ("phikon_v2_masked", "central"),
    # Add (extraction, token) pairs here for any other model/token combo
    # registered in EXTRACTIONS_DATA.
]

for extraction, token in REDUCTION_TARGETS:
    plot_reduction_grid(extraction, token)

## Joint embedding space across extractions/tokens

Concatenates several `(extraction, token)` sources (subsampled to
`n_per_source` cells each) into one matrix, runs PCA + UMAP jointly, and colors
the same 2D layout by `source` (which extraction/token each point came from),
by `wsi`, and by `cell_type` -- this is the direct "compare the embeddings
across multiple extractions" view: do masked-central tokens collapse into their
own cluster (they should, post mask_token-overwrite -- see the sanity check
above), does CLS drift WSI-to-WSI more than cell-type-to-cell-type, etc.

**To add another source**, append an `(extraction, token)` pair to
`JOINT_TARGETS`.

In [ ]:
JOINT_TARGETS = [
    ("phikon_v2", "cls"),
    ("phikon_v2", "central"),
    ("phikon_v2_masked", "cls"),
    ("phikon_v2_masked", "central"),
]


def joint_reduction(targets: list[tuple[str, str]], n_per_source: int | None = 5_000, seed: int = SEED):
    parts, metas = [], []
    for extraction, token in targets:
        data = EXTRACTIONS_DATA[extraction]
        meta_full = data["meta"]
        idx = subsample_index(len(meta_full), n_per_source, seed=seed)
        parts.append(data["tokens"][token][idx])
        m = meta_full.iloc[idx].copy()
        m["source"] = f"{extraction}/{token}"
        metas.append(m)
    emb_all = np.concatenate(parts, axis=0)
    meta_all = pd.concat(metas, ignore_index=True)
    pca_coords, umap_coords, var_ratio = reduce_2d(emb_all, seed=seed)
    return pca_coords, umap_coords, meta_all, var_ratio


pca_coords, umap_coords, meta_all, var_ratio = joint_reduction(JOINT_TARGETS)
rows = [("PCA", pca_coords)] + ([("UMAP", umap_coords)] if umap_coords is not None else [])

fig, axes = plt.subplots(len(rows), 3, figsize=(16, 4.5 * len(rows)), squeeze=False)
for r, (method, coords) in enumerate(rows):
    for c, color_key in enumerate(("source", "wsi", "cell_type")):
        scatter_colored(axes[r][c], coords, meta_all[color_key], f"{method} by {color_key}")
fig.suptitle("Joint embedding space across extractions/tokens")
fig.tight_layout()
plt.show()

## Extending this notebook

- **Another model/extraction**: add a key to `EXTRACTIONS` (its `h5_root` +
  `corner_keys`, i.e. the `patches_to_save` keys that model's
  `InferenceProvider` subclass saves, + whether it has a `cls` token) and
  re-run the loading cell -- it lands in `EXTRACTIONS_DATA` under that key.
- **Another token** beyond `cls`/the corner keys/`central`: both h5 files also
  carry boundary-pooled `embeddings_cell` / `embeddings_nucleus` (whole-cell /
  nucleus mean-pooled tokens, see `InferenceProvider.pool_boundary_tokens`) --
  just add `"cell"` / `"nucleus"` to the `token_keys` list built from
  `cfg["corner_keys"] + [...]` in the loading cell, or load them directly with
  `load_tokens(..., token_keys=["cell", "nucleus"])`.
- **Another cosine-similarity comparison**: append a dict to `COMPARISONS` --
  `a`/`b` are any two `(extraction, token)` pairs already in `EXTRACTIONS_DATA`.
- **Another PCA/UMAP panel**: append an `(extraction, token)` pair to
  `REDUCTION_TARGETS` (per-extraction grid) and/or `JOINT_TARGETS` (joint
  embedding space).